# Training code 

## Import

In [2]:
import aegnn
import pytorch_lightning as pl
import pytorch_lightning.loggers
import wandb
import datetime

from pathlib import Path

import os

In [3]:
os.environ["AEGNN_DATA_DIR"] = "/home/kevin-imagine/Documents/event_graph/data"
print(os.environ["AEGNN_DATA_DIR"])

/home/kevin-imagine/Documents/event_graph/data


## Classification

### Parameters

In [4]:
params = {"model":"graph_res",
          "task":"",
          "dim":3,
          "seed":12345,
          "lr": 1e-3,
          "weight_decay":5e-3,
          "eta_min":0.0,
          "max_epochs":100,
          "Trainer":{
            "max_epochs":100,
            "overfit_batches":0,
            "log_every_n_steps":10,
            "gradient_clip_val":0,
            "limit_train_batches":1,
            "limit_val_batches":1
          },
          "log-gradients":True,
          "profile":True,
          "debug":True,
          "gpu":0,
          "dataset":"ncars",
          "Data":{
              "batch_size":8,
              "num_workers":8,
              "pin_memory":False
          }          
}

### Model and dataset

In [5]:
data_module = aegnn.datasets.NCars(shuffle = True,
                                   **params["Data"],
                                   )

In [6]:
data_module.setup()

In [7]:
model = aegnn.models.RecognitionModel(params["model"],
                                      params["dataset"],
                                      num_classes=data_module.num_classes,
                                      img_shape=data_module.dims,
                                      dim = params["dim"],
                                      bias = True,
                                      root_weight = True,                                     
                                      )


### Loggers

In [8]:
log_settings = wandb.Settings(start_method='thread')
log_dir = "../training_log_aegnn/"
loggers = [aegnn.utils.loggers.LoggingLogger(None if params["debug"] else log_dir, name="debug")]

wandb: WARNING `start_method` is deprecated and will be removed in a future version of wandb. This setting is currently non-functional and safely ignored.


In [9]:
project = f"aegnn-{params['dataset']}-{params['task']}"
experiment_name = datetime.datetime.now().strftime("%Y%m%d%H%M%S")


In [10]:
if not params["debug"]:
    wandb_logger = pl.loggers.WandbLogger(project=project, save_dir=log_dir, settings=log_settings)
    if params["log-gradients"]:
        wandb_logger.watch(model, log="gradients")
    loggers.append(wandb_logger)
loggers = pl.loggers.LoggerCollection(loggers)

In [11]:
checkpoint_path = os.path.join(log_dir, "checkpoints", params["dataset"], params["task"], experiment_name)
Path(checkpoint_path).mkdir(parents=True,exist_ok=True)

In [12]:
callbacks = [
        pl.callbacks.LearningRateMonitor(),
        aegnn.utils.callbacks.BBoxLogger(classes=data_module.classes),
        # aegnn.utils.callbacks.PHyperLogger(args),
        aegnn.utils.callbacks.EpochLogger(),
        aegnn.utils.callbacks.FileLogger([model, model.model, data_module]),
        aegnn.utils.callbacks.FullModelCheckpoint(dirpath=checkpoint_path)
    ]

In [13]:
trainer_kwargs = {
    "gpus":[params["gpu"]] if params["gpu"] is not None else None,
    "profiler": "simple" if params["profile"] else False,
    "weights_summary": "full",
    "track_grad_norm": 2 if params["log-gradients"] else -1
}

In [14]:
trainer = pl.Trainer(logger=loggers,
                     callbacks=callbacks,
                     **params["Trainer"],
                     **trainer_kwargs
                     )

GPU available: True, used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs


In [15]:
trainer.fit(model, datamodule=data_module)

je vaius chercher ici: /home/kevin-imagine/Documents/event_graph/data/ncars/training/*
RAW FILer length 12961


100%|██████████| 12961/12961 [00:00<00:00, 176367.22it/s]

je vaius chercher ici: /home/kevin-imagine/Documents/event_graph/data/ncars/validation/*
RAW FILer length 2462



100%|██████████| 2462/2462 [00:00<00:00, 105706.65it/s]
/home/kevin-imagine/Documents/event_graph/.venv-gpu/lib/python3.10/site-packages/pytorch_lightning/core/datamodule.py:423: LightningDeprecationWarning: DataModule.setup has already been called, so it will not be called again. In v1.6 this behavior will change to always call DataModule.setup.
  rank_zero_deprecation(
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

   | Name                    | Type             | Params
--------------------------------------------------------------
0  | criterion               | CrossEntropyLoss | 0     
1  | model                   | GraphRes         | 30.4 K
2  | model.conv1             | SplineConv       | 80    
3  | model.conv1.aggr_module | MeanAggregation  | 0     
4  | model.conv1.lin         | Linear           | 8     
5  | model.norm1             | BatchNorm        | 16    
6  | model.norm1.module      | BatchNorm1d      | 16    
7  | model.conv2             | SplineConv       | 1.2 K 
8  | m

Validation sanity check: 0it [00:00, ?it/s]

...

Validating: 0it [00:00, ?it/s]

Validating: 0it [00:00, ?it/s]

Validating: 0it [00:00, ?it/s]

Validating: 0it [00:00, ?it/s]

FIT Profiler Report

Action                             	|  Mean duration (s)	|Num calls      	|  Total time (s) 	|  Percentage %   	|
--------------------------------------------------------------------------------------------------------------------------------------
Total                              	|  -              	|_              	|  234.79         	|  100 %          	|
--------------------------------------------------------------------------------------------------------------------------------------
run_training_epoch                 	|  2.0744         	|100            	|  207.44         	|  88.352         	|
get_train_batch                    	|  1.4969         	|100            	|  149.69         	|  63.753         	|
run_training_batch                 	|  0.23503        	|100            	|  23.503         	|  10.01          	|
optimizer_step_and_closure_0       	|  0.082498       	|100            	|  8.2498         	|  3.5137         	|
training_step_and_backward         

In [16]:
import torch

In [17]:
torch.cuda.is_available()

True

In [18]:
torch.cuda.get_device_name(0)

'NVIDIA RTX PRO 1000 Blackwell Generation Laptop GPU'